In [ ]:
import random
import sys
from pathlib import Path
from functools import partial

import numpy as np
import torch

from google.colab import drive
from pycocotools.coco import COCO
from torch.utils.data import DataLoader

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/ucu-ai-hw-2")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.data_prep import load_annotations
from scripts.data_cv import stratified_group_folds
from scripts.dataset_helpers import DetectionDataset, collate_data
from scripts.model_builders import build_ssd_model
from scripts.experiment_results import model_training_results
from scripts.experiment_runner import run_torch_cv

DIRECTORY = PROJECT_ROOT / "data"
INFO_TRAIN = DIRECTORY / "train" / "annotations_coco.json"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
FOLDS = 3
EPOCHS = 5
FRAME_STEP = 15
BATCH_SIZE = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
train_frames, train_boxes = load_annotations(COCO(str(INFO_TRAIN)), split="train")
frames = stratified_group_folds(train_frames, train_boxes, n_splits=FOLDS, seed=SEED)
annotations_by_image = {image_id: group for image_id, group in train_boxes.groupby("image_id")}
empty_annotations = train_boxes.iloc[:0].copy()

loading annotations into memory...
Done (t=15.98s)
creating index...
index created!


In [ ]:
cv_fold_loaders = {}

kwargs_dataset = {
    "directory": DIRECTORY,
    "annotations_by_image": annotations_by_image,
    "empty_annotations": empty_annotations,
}

for fold in range(FOLDS):
    training_frames = (frames.loc[frames[f"train_fold_{fold}"].eq(True)].sort_values(["video", "frame_id"])
        .groupby("video", group_keys=False).nth[::FRAME_STEP].reset_index(drop=True))

    validation_frames = frames.loc[frames["validation_fold"].eq(fold)].reset_index(drop=True)

    train_dataset = DetectionDataset(frames=training_frames, validation=False, **kwargs_dataset)
    valid_dataset = DetectionDataset(frames=validation_frames, validation=True, **kwargs_dataset)

    cv_fold_loaders[fold] = {
        "train": DataLoader(train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=0,
            collate_fn=collate_data
        ),
        "valid": DataLoader(
            valid_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=0,
            collate_fn=collate_data
        )
    }

    print(f"Fold {fold}: {len(train_dataset)} training frames, {len(valid_dataset)} validation frames")

Fold 0: 1107 training frames, 7943 validation frames
Fold 1: 1094 training frames, 8088 validation frames
Fold 2: 1091 training frames, 8170 validation frames


## SSD300 VGG16 baseline model - iou lowered

In [ ]:
iou_histories, iou_results = run_torch_cv(cv_fold_loaders=cv_fold_loaders,
    model_method=partial(build_ssd_model, iou_threshold=0.4), output_dir=PROJECT_ROOT / "outputs" / "ssd_iou_04",
    annotations_path=INFO_TRAIN, device=device, seed=SEED, epochs=EPOCHS)

loading annotations into memory...
Done (t=5.31s)
creating index...
index created!
Downloading: "https://download.pytorch.org/models/ssd300_vgg16_coco-b556d3b4.pth" to /root/.cache/torch/hub/checkpoints/ssd300_vgg16_coco-b556d3b4.pth


100%|██████████| 136M/136M [00:00<00:00, 174MB/s]


Loading and preparing results...
DONE (t=5.92s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=371.90s).
Accumulating evaluation results...
DONE (t=15.04s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.027
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.066
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.020
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.003
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.022
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.088
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.015
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.037
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.052
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets

In [ ]:
iou_results_wrap = model_training_results(ROOT=PROJECT_ROOT, filename="ssd_iou_04", cv_folds=range(FOLDS))


Fold 0


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,5.824788,0.0001,0.026760,0.066048,0.002887
1,2,4.300269,0.0001,0.031470,0.072097,0.004277
2,3,3.920690,0.0001,0.037923,0.088650,0.004322
3,4,3.599671,0.0001,0.041017,0.094176,0.005443
4,5,3.421155,0.0001,0.031834,0.075623,0.005482



Fold 1


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,6.100469,0.0001,0.035143,0.083136,0.004752
1,2,4.488857,0.0001,0.044125,0.096314,0.004671
2,3,4.023324,0.0001,0.051700,0.122898,0.005229
3,4,3.686197,0.0001,0.054860,0.124995,0.005152
4,5,3.436188,0.0001,0.060336,0.136415,0.005699



Fold 2


,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,1,5.914107,0.0001,0.021772,0.055657,0.004231
1,2,4.424781,0.0001,0.028853,0.075709,0.004941
2,3,3.974383,0.0001,0.037233,0.094425,0.007652
3,4,3.706435,0.0001,0.037478,0.094150,0.005615
4,5,3.459713,0.0001,0.037774,0.097184,0.006105



Best results per fold


,fold,epoch,train_loss,learning_rate,mAP,AP50,AP_small
0,0,4,3.599671,0.0001,0.041017,0.094176,0.005443
1,1,5,3.436188,0.0001,0.060336,0.136415,0.005699
2,2,5,3.459713,0.0001,0.037774,0.097184,0.006105



Metrics mean and std across folds


,mAP,AP50,AP_small
mean,0.046375,0.109258,0.005749
std,0.012198,0.023566,0.000334
